In [2]:
!pip install -q ultralytics mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.1 MB/s eta 0:00:00


In [3]:
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [14]:
import cv2
import json
import time
import numpy as np
from collections import deque
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
YOLO_MODEL_PATH = "/content/yolo_dms.onnx"
CLASS_JSON_PATH = "/content/yolo_classes.json"
INPUT_VIDEO_PATH  = "/content/headturn.mp4"
OUTPUT_VIDEO_PATH = "/content/output.mp4"

BUFFER_WINDOW = 60
EAR_THRESHOLD   = 0.25
MAR_THRESHOLD   = 0.35
YAW_THRESHOLD   = 25
PITCH_THRESHOLD = 35
EMA_ALPHA = 0.30

# ──────────────────────────────────────────────────────────────────────────────
# INITIALIZATION (MODERN TASKS API)
# ──────────────────────────────────────────────────────────────────────────────
with open(CLASS_JSON_PATH, 'r') as f:
    class_names = json.load(f)

yolo_model = YOLO(YOLO_MODEL_PATH, task='detect')

# 1. Load the Task Model we just downloaded
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')

# 2. Configure the Face Landmarker
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

# 3. Create the detector
detector = vision.FaceLandmarker.create_from_options(options)

ear_buffer       = deque(maxlen=BUFFER_WINDOW)
mar_buffer       = deque(maxlen=BUFFER_WINDOW)
gaze_buffer      = deque(maxlen=BUFFER_WINDOW)
phone_buffer     = deque(maxlen=BUFFER_WINDOW)
ingestion_buffer = deque(maxlen=BUFFER_WINDOW)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH = [78, 308, 13, 14]

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, 330.0, -65.0),         # Chin (Positive Y is down)
    (-225.0, -170.0, -135.0),    # Left eye corner (Negative Y is up)
    (225.0, -170.0, -135.0),     # Right eye corner
    (-150.0, 150.0, -125.0),     # Left mouth corner
    (150.0, 150.0, -125.0),      # Right mouth corner
], dtype=np.float32)

_ema_pitch = None
_ema_yaw = None

# ──────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS (Zero changes required)
# ──────────────────────────────────────────────────────────────────────────────
def calculate_ear(landmarks, eye_indices, img_w, img_h):
    pts = [np.array([landmarks[i].x * img_w, landmarks[i].y * img_h]) for i in eye_indices]
    p2_p6 = np.linalg.norm(pts[1] - pts[5])
    p3_p5 = np.linalg.norm(pts[2] - pts[4])
    p1_p4 = np.linalg.norm(pts[0] - pts[3])
    return (p2_p6 + p3_p5) / (2.0 * p1_p4)

def calculate_mar(landmarks, mouth_indices, img_w, img_h):
    p_left = np.array([landmarks[mouth_indices[0]].x * img_w, landmarks[mouth_indices[0]].y * img_h])
    p_right = np.array([landmarks[mouth_indices[1]].x * img_w, landmarks[mouth_indices[1]].y * img_h])
    p_top = np.array([landmarks[mouth_indices[2]].x * img_w, landmarks[mouth_indices[2]].y * img_h])
    p_bottom = np.array([landmarks[mouth_indices[3]].x * img_w, landmarks[mouth_indices[3]].y * img_h])
    horizontal = np.linalg.norm(p_left - p_right)
    vertical   = np.linalg.norm(p_top - p_bottom)
    return vertical / horizontal if horizontal > 0 else 0.0

def estimate_head_pose(landmarks, img_w, img_h):
    global _ema_pitch, _ema_yaw
    image_points = np.array([
        (landmarks[1].x   * img_w, landmarks[1].y   * img_h),
        (landmarks[152].x * img_w, landmarks[152].y * img_h),
        (landmarks[33].x  * img_w, landmarks[33].y  * img_h),
        (landmarks[263].x * img_w, landmarks[263].y * img_h),
        (landmarks[61].x  * img_w, landmarks[61].y  * img_h),
        (landmarks[291].x * img_w, landmarks[291].y * img_h),
    ], dtype=np.float32)

    focal_length = float(img_w)
    cam_matrix = np.array([[focal_length, 0, img_w / 2.0], [0, focal_length, img_h / 2.0], [0, 0, 1.0]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1), dtype=np.float32)

    success, rot_vec, trans_vec = cv2.solvePnP(MODEL_POINTS, image_points, cam_matrix, dist_coeffs, flags=cv2.SOLVEPNP_SQPNP)
    if not success:
        return (_ema_pitch if _ema_pitch is not None else 0.0, _ema_yaw if _ema_yaw is not None else 0.0)

    rot_mat, _ = cv2.Rodrigues(rot_vec)
    proj_matrix = np.hstack((rot_mat, trans_vec))
    _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)

    raw_pitch = float(euler_angles[0][0])
    raw_yaw   = -float(euler_angles[1][0])

    if _ema_pitch is None:
        _ema_pitch, _ema_yaw = raw_pitch, raw_yaw
    else:
        _ema_pitch = EMA_ALPHA * raw_pitch + (1 - EMA_ALPHA) * _ema_pitch
        _ema_yaw   = EMA_ALPHA * raw_yaw + (1 - EMA_ALPHA) * _ema_yaw
    return _ema_pitch, _ema_yaw

# ──────────────────────────────────────────────────────────────────────────────
# HUD
# ──────────────────────────────────────────────────────────────────────────────
_NORMAL_ALERT = ("SYSTEM NORMAL: DRIVER ATTENTIVE", (0, 255, 0), -1)
_HUD_FONT = cv2.FONT_HERSHEY_SIMPLEX
_HUD_FONT_SCALE = 0.65
_HUD_THICKNESS = 2
_HUD_LINE_H = 30
_HUD_STATS_Y = 18
_HUD_FIRST_Y = 48

def render_hud(frame, alerts, pitch, yaw, fps):
    if not alerts:
        alerts = [_NORMAL_ALERT]
    alerts = sorted(alerts, key=lambda x: x[2], reverse=True)
    n_rows = len(alerts)
    banner_h = _HUD_FIRST_Y + n_rows * _HUD_LINE_H + 8

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], banner_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
    cv2.putText(frame, f"Pitch: {pitch:+.1f}  Yaw: {yaw:+.1f}  Process FPS: {fps:.0f}", (20, _HUD_STATS_Y), _HUD_FONT, 0.45, (210, 210, 210), 1, cv2.LINE_AA)

    for idx, (text, colour, _) in enumerate(alerts):
        y = _HUD_FIRST_Y + idx * _HUD_LINE_H
        cv2.putText(frame, text, (20, y), _HUD_FONT, _HUD_FONT_SCALE, colour, _HUD_THICKNESS, cv2.LINE_AA)

# ──────────────────────────────────────────────────────────────────────────────
# VIDEO PROCESSING
# ──────────────────────────────────────────────────────────────────────────────
def process_video(input_path, output_path):
    print(f"Loading video: {input_path}")
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print("Error: Could not open video file. Did you upload IPCV_data.mp4 to Colab?")
        return

    fps_input = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

    prev_frame_time = time.perf_counter()
    frame_count = 0

    print(f"Processing {total_frames} frames...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        current_time = time.perf_counter()
        fps = 1 / (current_time - prev_frame_time) if (current_time - prev_frame_time) > 0 else 0
        prev_frame_time = current_time

        h, w, _ = frame.shape
        display_frame = frame.copy()

        # 1. Convert frame for MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 2. Run the modern detector
        detection_result = detector.detect(mp_image)

        is_drowsy, is_yawning, is_distracted_gaze = False, False, False
        current_pitch = _ema_pitch if _ema_pitch is not None else 0.0
        current_yaw = _ema_yaw if _ema_yaw is not None else 0.0

        # 3. Process the flat list of landmarks
        if detection_result.face_landmarks:
            landmarks = detection_result.face_landmarks[0]

            left_ear = calculate_ear(landmarks, LEFT_EYE, w, h)
            right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0
            ear_buffer.append(avg_ear < EAR_THRESHOLD)

            current_mar = calculate_mar(landmarks, MOUTH, w, h)
            mar_buffer.append(current_mar > MAR_THRESHOLD)

            current_pitch, current_yaw = estimate_head_pose(landmarks, w, h)

            if sum(ear_buffer) >= int(0.8 * BUFFER_WINDOW): is_drowsy = True
            if sum(mar_buffer) >= int(0.6 * BUFFER_WINDOW): is_yawning = True

            gaze_distracted = (abs(current_yaw) > YAW_THRESHOLD or current_pitch > PITCH_THRESHOLD)
            gaze_buffer.append(gaze_distracted)
            if sum(gaze_buffer) >= int(0.7 * BUFFER_WINDOW): is_distracted_gaze = True
        else:
            ear_buffer.append(True)
            gaze_buffer.append(True)
            if sum(ear_buffer) >= int(0.8 * BUFFER_WINDOW): is_drowsy = True
            if sum(gaze_buffer) >= int(0.7 * BUFFER_WINDOW): is_distracted_gaze = True

        yolo_results = yolo_model.predict(source=frame, conf=0.55, verbose=False)
        frame_has_phone = False
        frame_has_ingestion = False

        for box in yolo_results[0].boxes:
            cls_id = int(box.cls[0])
            cls_name = class_names.get(str(cls_id), "unknown")
            if cls_name in ("face", "seatbelt"): continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf_val = float(box.conf[0])

            if cls_name == "phone":
                frame_has_phone = True
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(display_frame, f"PHONE {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            elif cls_name in ("drinking", "eating", "cigarette"):
                frame_has_ingestion = True
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                cv2.putText(display_frame, f"INGESTION {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 165, 255), 2, cv2.LINE_AA)

        phone_buffer.append(frame_has_phone)
        ingestion_buffer.append(frame_has_ingestion)
        # Now requires 18 frames out of 60 (approx. 0.6 seconds of sustained detection)
        detected_phone = sum(phone_buffer) >= int(0.3 * BUFFER_WINDOW)
        detected_ingestion = sum(ingestion_buffer) >= int(0.3 * BUFFER_WINDOW)

        active_alerts = []
        if is_drowsy:
            active_alerts.append(("CRITICAL: DROWSINESS DETECTED", (0, 0, 255), 2))
        if detected_phone:
            if current_pitch < -5:
                active_alerts.append(("CRITICAL: TEXTING & DRIVING", (0, 0, 255), 2))
            else:
                active_alerts.append(("WARNING: CELLPHONE DISTRACTION", (0, 165, 255), 1))
        if is_distracted_gaze:
            active_alerts.append(("WARNING: DRIVER GAZE DISTRACTED", (0, 255, 255), 1))
        if detected_ingestion:
            active_alerts.append(("WARNING: INGESTION DISTRACTION", (0, 165, 255), 1))
        if is_yawning:
            active_alerts.append(("ADVISORY: DRIVER FATIGUE YAWN", (255, 255, 0), 0))

        render_hud(display_frame, active_alerts, current_pitch, current_yaw, fps)
        out.write(display_frame)

        if frame_count % 50 == 0:
            print(f"Processed {frame_count}/{total_frames} frames...")

    cap.release()
    out.release()
    print(f"\n✅ Processing complete! Download your processed video from the Colab file browser: {output_path}")

process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Loading video: /content/headturn.mp4
Processing 551 frames...
Loading /content/yolo_dms.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider
Processed 50/551 frames...
Processed 100/551 frames...
Processed 150/551 frames...
Processed 200/551 frames...
Processed 250/551 frames...
Processed 300/551 frames...
Processed 350/551 frames...
Processed 400/551 frames...
Processed 450/551 frames...
Processed 500/551 frames...
Processed 550/551 frames...

✅ Processing complete! Download your processed video from the Colab file browser: /content/output.mp4


In [4]:
import cv2
import json
import time
import numpy as np
from collections import deque
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
YOLO_MODEL_PATH = "/content/yolo_dms.onnx"
CLASS_JSON_PATH = "/content/yolo_classes.json"
INPUT_VIDEO_PATH  = "/content/headturn.mp4"
OUTPUT_VIDEO_PATH = "/content/output.mp4"

BUFFER_WINDOW = 60
EAR_THRESHOLD   = 0.25
MAR_THRESHOLD   = 0.35
YAW_THRESHOLD   = 25
PITCH_THRESHOLD = 35
EMA_ALPHA = 0.30

# ──────────────────────────────────────────────────────────────────────────────
# INITIALIZATION (MODERN TASKS API)
# ──────────────────────────────────────────────────────────────────────────────
with open(CLASS_JSON_PATH, 'r') as f:
    class_names = json.load(f)

yolo_model = YOLO(YOLO_MODEL_PATH, task='detect')

# 1. Load the Task Model we just downloaded
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')

# 2. Configure the Face Landmarker
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

# 3. Create the detector
detector = vision.FaceLandmarker.create_from_options(options)

ear_buffer       = deque(maxlen=BUFFER_WINDOW)
mar_buffer       = deque(maxlen=BUFFER_WINDOW)
gaze_buffer      = deque(maxlen=BUFFER_WINDOW)
phone_buffer     = deque(maxlen=BUFFER_WINDOW)
ingestion_buffer = deque(maxlen=BUFFER_WINDOW)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH = [78, 308, 13, 14]

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, 330.0, -65.0),         # Chin (Positive Y is down)
    (-225.0, -170.0, -135.0),    # Left eye corner (Negative Y is up)
    (225.0, -170.0, -135.0),     # Right eye corner
    (-150.0, 150.0, -125.0),     # Left mouth corner
    (150.0, 150.0, -125.0),      # Right mouth corner
], dtype=np.float32)

_ema_pitch = None
_ema_yaw = None

# ──────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS (Zero changes required)
# ──────────────────────────────────────────────────────────────────────────────
def calculate_ear(landmarks, eye_indices, img_w, img_h):
    pts = [np.array([landmarks[i].x * img_w, landmarks[i].y * img_h]) for i in eye_indices]
    p2_p6 = np.linalg.norm(pts[1] - pts[5])
    p3_p5 = np.linalg.norm(pts[2] - pts[4])
    p1_p4 = np.linalg.norm(pts[0] - pts[3])
    return (p2_p6 + p3_p5) / (2.0 * p1_p4)

def calculate_mar(landmarks, mouth_indices, img_w, img_h):
    p_left = np.array([landmarks[mouth_indices[0]].x * img_w, landmarks[mouth_indices[0]].y * img_h])
    p_right = np.array([landmarks[mouth_indices[1]].x * img_w, landmarks[mouth_indices[1]].y * img_h])
    p_top = np.array([landmarks[mouth_indices[2]].x * img_w, landmarks[mouth_indices[2]].y * img_h])
    p_bottom = np.array([landmarks[mouth_indices[3]].x * img_w, landmarks[mouth_indices[3]].y * img_h])
    horizontal = np.linalg.norm(p_left - p_right)
    vertical   = np.linalg.norm(p_top - p_bottom)
    return vertical / horizontal if horizontal > 0 else 0.0

def estimate_head_pose(landmarks, img_w, img_h):
    global _ema_pitch, _ema_yaw
    image_points = np.array([
        (landmarks[1].x   * img_w, landmarks[1].y   * img_h),
        (landmarks[152].x * img_w, landmarks[152].y * img_h),
        (landmarks[33].x  * img_w, landmarks[33].y  * img_h),
        (landmarks[263].x * img_w, landmarks[263].y * img_h),
        (landmarks[61].x  * img_w, landmarks[61].y  * img_h),
        (landmarks[291].x * img_w, landmarks[291].y * img_h),
    ], dtype=np.float32)

    focal_length = float(img_w)
    cam_matrix = np.array([[focal_length, 0, img_w / 2.0], [0, focal_length, img_h / 2.0], [0, 0, 1.0]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1), dtype=np.float32)

    success, rot_vec, trans_vec = cv2.solvePnP(MODEL_POINTS, image_points, cam_matrix, dist_coeffs, flags=cv2.SOLVEPNP_SQPNP)
    if not success:
        return (_ema_pitch if _ema_pitch is not None else 0.0, _ema_yaw if _ema_yaw is not None else 0.0)

    rot_mat, _ = cv2.Rodrigues(rot_vec)
    proj_matrix = np.hstack((rot_mat, trans_vec))
    _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)

    raw_pitch = float(euler_angles[0][0])
    raw_yaw   = -float(euler_angles[1][0])

    if _ema_pitch is None:
        _ema_pitch, _ema_yaw = raw_pitch, raw_yaw
    else:
        _ema_pitch = EMA_ALPHA * raw_pitch + (1 - EMA_ALPHA) * _ema_pitch
        _ema_yaw   = EMA_ALPHA * raw_yaw + (1 - EMA_ALPHA) * _ema_yaw
    return _ema_pitch, _ema_yaw

# ──────────────────────────────────────────────────────────────────────────────
# HUD
# ──────────────────────────────────────────────────────────────────────────────
_NORMAL_ALERT = ("SYSTEM NORMAL: DRIVER ATTENTIVE", (0, 255, 0), -1)
_HUD_FONT = cv2.FONT_HERSHEY_SIMPLEX
_HUD_FONT_SCALE = 0.65
_HUD_THICKNESS = 2
_HUD_LINE_H = 30
_HUD_STATS_Y = 18
_HUD_FIRST_Y = 48

def render_hud(frame, alerts, pitch, yaw, mar, fps):
    if not alerts:
        alerts = [_NORMAL_ALERT]
    alerts = sorted(alerts, key=lambda x: x[2], reverse=True)
    n_rows = len(alerts)
    banner_h = _HUD_FIRST_Y + n_rows * _HUD_LINE_H + 8

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], banner_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)

    # Added MAR to the HUD stats for easy tuning
    cv2.putText(frame, f"Pitch: {pitch:+.1f}  Yaw: {yaw:+.1f}  MAR: {mar:.2f}  FPS: {fps:.0f}", (20, _HUD_STATS_Y), _HUD_FONT, 0.45, (210, 210, 210), 1, cv2.LINE_AA)

    for idx, (text, colour, _) in enumerate(alerts):
        y = _HUD_FIRST_Y + idx * _HUD_LINE_H
        cv2.putText(frame, text, (20, y), _HUD_FONT, _HUD_FONT_SCALE, colour, _HUD_THICKNESS, cv2.LINE_AA)

# ──────────────────────────────────────────────────────────────────────────────
# VIDEO PROCESSING
# ──────────────────────────────────────────────────────────────────────────────
def process_video(input_path, output_path):
    print(f"Loading video: {input_path}")
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print("Error: Could not open video file. Did you upload IPCV_data.mp4 to Colab?")
        return

    fps_input = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

    prev_frame_time = time.perf_counter()
    frame_count = 0

    print(f"Processing {total_frames} frames...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        current_time = time.perf_counter()
        fps = 1 / (current_time - prev_frame_time) if (current_time - prev_frame_time) > 0 else 0
        prev_frame_time = current_time

        h, w, _ = frame.shape
        display_frame = frame.copy()

        # 1. Convert frame for MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 2. Run the modern detector
        detection_result = detector.detect(mp_image)

        is_drowsy, is_yawning, is_distracted_gaze = False, False, False
        current_pitch = _ema_pitch if _ema_pitch is not None else 0.0
        current_yaw = _ema_yaw if _ema_yaw is not None else 0.0
        current_mar = 0.0
        mouth_x, mouth_y = 0, 0  # Spatial fusion coordinates

# 3. Process the flat list of landmarks
        if detection_result.face_landmarks:
            landmarks = detection_result.face_landmarks[0]

            # Extract spatial coordinate of the inner top lip for YOLO fusion
            mouth_x = int(landmarks[13].x * w)
            mouth_y = int(landmarks[13].y * h)

            # CALCULATE GAZE FIRST
            current_pitch, current_yaw = estimate_head_pose(landmarks, w, h)
            gaze_distracted = (abs(current_yaw) > YAW_THRESHOLD or current_pitch > PITCH_THRESHOLD)
            gaze_buffer.append(gaze_distracted)
            if sum(gaze_buffer) >= int(0.7 * BUFFER_WINDOW): is_distracted_gaze = True

            # STATE MACHINE GATE: Only evaluate Drowsy/Yawn if the driver is looking forward
            if not gaze_distracted:
                left_ear = calculate_ear(landmarks, LEFT_EYE, w, h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                ear_buffer.append(avg_ear < EAR_THRESHOLD)

                current_mar = calculate_mar(landmarks, MOUTH, w, h)
                mar_buffer.append(current_mar > MAR_THRESHOLD)
            else:
                # Suspend tracking: Prevent "Perspective Collapse" false positives
                ear_buffer.append(False)
                mar_buffer.append(False)
                current_mar = 0.0 # Zero out for the HUD display

            if sum(ear_buffer) >= int(0.8 * BUFFER_WINDOW): is_drowsy = True
            if sum(mar_buffer) >= int(0.6 * BUFFER_WINDOW): is_yawning = True

        else:
            ear_buffer.append(True)
            gaze_buffer.append(True)
            if sum(ear_buffer) >= int(0.8 * BUFFER_WINDOW): is_drowsy = True
            if sum(gaze_buffer) >= int(0.7 * BUFFER_WINDOW): is_distracted_gaze = True

        yolo_results = yolo_model.predict(source=frame, conf=0.55, verbose=False)
        frame_has_phone = False
        frame_has_ingestion = False

        for box in yolo_results[0].boxes:
            cls_id = int(box.cls[0])
            cls_name = class_names.get(str(cls_id), "unknown")
            if cls_name in ("face", "seatbelt"): continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf_val = float(box.conf[0])

            if cls_name == "phone":
                frame_has_phone = True
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(display_frame, f"PHONE {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            elif cls_name in ("drinking", "eating", "cigarette"):
                # CROSS-MODAL FUSION: Reject false positives like fixing glasses/earphones
                is_near_mouth = False

                # If MediaPipe sees a face, enforce the spatial constraint
                if mouth_x > 0 and mouth_y > 0:
                    padding = 60 # 60-pixel forgiveness radius around the mouth
                    # Check if the mouth falls inside the YOLO bounding box (plus padding)
                    if (x1 - padding) <= mouth_x <= (x2 + padding) and (y1 - padding) <= mouth_y <= (y2 + padding):
                        is_near_mouth = True
                else:
                    # If MediaPipe lost the face entirely, trust YOLO as a fallback
                    is_near_mouth = True

                if is_near_mouth:
                    frame_has_ingestion = True
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                    cv2.putText(display_frame, f"INGESTION {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 165, 255), 2, cv2.LINE_AA)

        phone_buffer.append(frame_has_phone)
        ingestion_buffer.append(frame_has_ingestion)
        # Now requires 18 frames out of 60 (approx. 0.6 seconds of sustained detection)
        detected_phone = sum(phone_buffer) >= int(0.3 * BUFFER_WINDOW)
        detected_ingestion = sum(ingestion_buffer) >= int(0.3 * BUFFER_WINDOW)

        active_alerts = []
        if is_drowsy:
            active_alerts.append(("CRITICAL: DROWSINESS DETECTED", (0, 0, 255), 2))
        if detected_phone:
            if current_pitch < -5:
                active_alerts.append(("CRITICAL: TEXTING & DRIVING", (0, 0, 255), 2))
            else:
                active_alerts.append(("WARNING: CELLPHONE DISTRACTION", (0, 165, 255), 1))
        if is_distracted_gaze:
            active_alerts.append(("WARNING: DRIVER GAZE DISTRACTED", (0, 255, 255), 1))
        if detected_ingestion:
            active_alerts.append(("WARNING: INGESTION DISTRACTION", (0, 165, 255), 1))
        if is_yawning:
            active_alerts.append(("ADVISORY: DRIVER FATIGUE YAWN", (255, 255, 0), 0))

        # Pass current_mar to the updated render_hud function
        render_hud(display_frame, active_alerts, current_pitch, current_yaw, current_mar, fps)
        out.write(display_frame)

        if frame_count % 50 == 0:
            print(f"Processed {frame_count}/{total_frames} frames...")

    cap.release()
    out.release()
    print(f"\n✅ Processing complete! Download your processed video from the Colab file browser: {output_path}")

process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading video: /content/headturn.mp4
Processing 551 frames...
Loading /content/yolo_dms.onnx for ONNX Runtime inference...
requirements: Ultralytics requirements ['onnx', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 8 packages in 293ms
Prepared 2 packages in 3.79s
Installed 2 packages in 275ms
 + onnx==1.21.0
 + onnxruntime-gpu==1.26.0

requirements: AutoUpdate success ✅ 4.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Using ONNX Runtime 1.26.0 with CUDAExecutionProvider
Processed 50/551 frames...
Processed 100/551 frames...
Processed 150/551 frames...
Processed 200/551 frames...

In [6]:
import cv2
import json
import time
import numpy as np
from collections import deque
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
YOLO_MODEL_PATH = "/content/yolo_dms.onnx"
CLASS_JSON_PATH = "/content/yolo_classes.json"
INPUT_VIDEO_PATH  = "/content/headturn.mp4"
OUTPUT_VIDEO_PATH = "/content/output.mp4"

BUFFER_WINDOW = 60
EAR_THRESHOLD   = 0.25
MAR_THRESHOLD   = 0.35
YAW_THRESHOLD   = 25
PITCH_THRESHOLD = 35
EMA_ALPHA = 0.30

# ──────────────────────────────────────────────────────────────────────────────
# INITIALIZATION (MODERN TASKS API)
# ──────────────────────────────────────────────────────────────────────────────
with open(CLASS_JSON_PATH, 'r') as f:
    class_names = json.load(f)

yolo_model = YOLO(YOLO_MODEL_PATH, task='detect')

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

ear_buffer       = deque(maxlen=BUFFER_WINDOW)
mar_buffer       = deque(maxlen=BUFFER_WINDOW)
gaze_buffer      = deque(maxlen=BUFFER_WINDOW)
phone_buffer     = deque(maxlen=BUFFER_WINDOW)
ingestion_buffer = deque(maxlen=BUFFER_WINDOW)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH = [78, 308, 13, 14]

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, 330.0, -65.0),         # Chin (Positive Y is down)
    (-225.0, -170.0, -135.0),    # Left eye corner (Negative Y is up)
    (225.0, -170.0, -135.0),     # Right eye corner
    (-150.0, 150.0, -125.0),     # Left mouth corner
    (150.0, 150.0, -125.0),      # Right mouth corner
], dtype=np.float32)

_ema_pitch = None
_ema_yaw = None

# ──────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
def calculate_ear(landmarks, eye_indices, img_w, img_h):
    pts = [np.array([landmarks[i].x * img_w, landmarks[i].y * img_h]) for i in eye_indices]
    p2_p6 = np.linalg.norm(pts[1] - pts[5])
    p3_p5 = np.linalg.norm(pts[2] - pts[4])
    p1_p4 = np.linalg.norm(pts[0] - pts[3])
    return (p2_p6 + p3_p5) / (2.0 * p1_p4)

def calculate_mar(landmarks, mouth_indices, img_w, img_h):
    p_left = np.array([landmarks[mouth_indices[0]].x * img_w, landmarks[mouth_indices[0]].y * img_h])
    p_right = np.array([landmarks[mouth_indices[1]].x * img_w, landmarks[mouth_indices[1]].y * img_h])
    p_top = np.array([landmarks[mouth_indices[2]].x * img_w, landmarks[mouth_indices[2]].y * img_h])
    p_bottom = np.array([landmarks[mouth_indices[3]].x * img_w, landmarks[mouth_indices[3]].y * img_h])
    horizontal = np.linalg.norm(p_left - p_right)
    vertical   = np.linalg.norm(p_top - p_bottom)
    return vertical / horizontal if horizontal > 0 else 0.0

def estimate_head_pose(landmarks, img_w, img_h):
    global _ema_pitch, _ema_yaw
    image_points = np.array([
        (landmarks[1].x   * img_w, landmarks[1].y   * img_h),
        (landmarks[152].x * img_w, landmarks[152].y * img_h),
        (landmarks[33].x  * img_w, landmarks[33].y  * img_h),
        (landmarks[263].x * img_w, landmarks[263].y * img_h),
        (landmarks[61].x  * img_w, landmarks[61].y  * img_h),
        (landmarks[291].x * img_w, landmarks[291].y * img_h),
    ], dtype=np.float32)

    focal_length = float(img_w)
    cam_matrix = np.array([[focal_length, 0, img_w / 2.0], [0, focal_length, img_h / 2.0], [0, 0, 1.0]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1), dtype=np.float32)

    success, rot_vec, trans_vec = cv2.solvePnP(MODEL_POINTS, image_points, cam_matrix, dist_coeffs, flags=cv2.SOLVEPNP_SQPNP)
    if not success:
        return (_ema_pitch if _ema_pitch is not None else 0.0, _ema_yaw if _ema_yaw is not None else 0.0)

    rot_mat, _ = cv2.Rodrigues(rot_vec)
    proj_matrix = np.hstack((rot_mat, trans_vec))
    _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)

    raw_pitch = float(euler_angles[0][0])
    raw_yaw   = -float(euler_angles[1][0])

    if _ema_pitch is None:
        _ema_pitch, _ema_yaw = raw_pitch, raw_yaw
    else:
        _ema_pitch = EMA_ALPHA * raw_pitch + (1 - EMA_ALPHA) * _ema_pitch
        _ema_yaw   = EMA_ALPHA * raw_yaw + (1 - EMA_ALPHA) * _ema_yaw
    return _ema_pitch, _ema_yaw

# ──────────────────────────────────────────────────────────────────────────────
# HUD
# ──────────────────────────────────────────────────────────────────────────────
_NORMAL_ALERT = ("SYSTEM NORMAL: DRIVER ATTENTIVE", (0, 255, 0), -1)
_HUD_FONT = cv2.FONT_HERSHEY_SIMPLEX
_HUD_FONT_SCALE = 0.65
_HUD_THICKNESS = 2
_HUD_LINE_H = 30
_HUD_STATS_Y = 18
_HUD_FIRST_Y = 48

def render_hud(frame, alerts, pitch, yaw, mar, fps):
    if not alerts:
        alerts = [_NORMAL_ALERT]
    alerts = sorted(alerts, key=lambda x: x[2], reverse=True)
    n_rows = len(alerts)
    banner_h = _HUD_FIRST_Y + n_rows * _HUD_LINE_H + 8

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], banner_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
    cv2.putText(frame, f"Pitch: {pitch:+.1f}  Yaw: {yaw:+.1f}  MAR: {mar:.2f}  FPS: {fps:.0f}", (20, _HUD_STATS_Y), _HUD_FONT, 0.45, (210, 210, 210), 1, cv2.LINE_AA)

    for idx, (text, colour, _) in enumerate(alerts):
        y = _HUD_FIRST_Y + idx * _HUD_LINE_H
        cv2.putText(frame, text, (20, y), _HUD_FONT, _HUD_FONT_SCALE, colour, _HUD_THICKNESS, cv2.LINE_AA)

# ──────────────────────────────────────────────────────────────────────────────
# VIDEO PROCESSING
# ──────────────────────────────────────────────────────────────────────────────
def process_video(input_path, output_path):
    print(f"Loading video: {input_path}")
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print("Error: Could not open video file.")
        return

    fps_input = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

    prev_frame_time = time.perf_counter()
    frame_count = 0

    print(f"Processing {total_frames} frames...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        current_time = time.perf_counter()
        fps = 1 / (current_time - prev_frame_time) if (current_time - prev_frame_time) > 0 else 0
        prev_frame_time = current_time

        h, w, _ = frame.shape
        display_frame = frame.copy()

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = detector.detect(mp_image)

        is_drowsy, is_yawning, is_distracted_gaze = False, False, False
        current_pitch = _ema_pitch if _ema_pitch is not None else 0.0
        current_yaw = _ema_yaw if _ema_yaw is not None else 0.0
        current_mar = 0.0
        mouth_x, mouth_y = 0, 0

        # --- 1. MEASUREMENT PHASE ---
        if detection_result.face_landmarks:
            landmarks = detection_result.face_landmarks[0]

            mouth_x = int(landmarks[13].x * w)
            mouth_y = int(landmarks[13].y * h)

            current_pitch, current_yaw = estimate_head_pose(landmarks, w, h)
            gaze_is_bad = (abs(current_yaw) > YAW_THRESHOLD or current_pitch > PITCH_THRESHOLD)
            gaze_buffer.append(gaze_is_bad)

            if not gaze_is_bad:
                # Face visible and forward -> Normal Tracking
                left_ear = calculate_ear(landmarks, LEFT_EYE, w, h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                ear_buffer.append(avg_ear < EAR_THRESHOLD)

                current_mar = calculate_mar(landmarks, MOUTH, w, h)
                mar_buffer.append(current_mar > MAR_THRESHOLD)
            else:
                # Face visible but turned -> SUSPEND tracking (Prevent Perspective Collapse)
                ear_buffer.append(False)
                mar_buffer.append(False)
        else:
            # SENSOR DROPOUT: Face completely lost -> Force gaze, suspend tracking
            gaze_buffer.append(True)
            ear_buffer.append(False)
            mar_buffer.append(False)

        # --- 2. VOTING PHASE ---
        if sum(gaze_buffer) >= int(0.7 * BUFFER_WINDOW): is_distracted_gaze = True
        if sum(ear_buffer) >= int(0.8 * BUFFER_WINDOW): is_drowsy = True
        if sum(mar_buffer) >= int(0.6 * BUFFER_WINDOW): is_yawning = True

        # --- 3. YOLO PHASE ---
        yolo_results = yolo_model.predict(source=frame, conf=0.55, verbose=False)
        frame_has_phone = False
        frame_has_ingestion = False

        for box in yolo_results[0].boxes:
            cls_id = int(box.cls[0])
            cls_name = class_names.get(str(cls_id), "unknown")
            if cls_name in ("face", "seatbelt"): continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf_val = float(box.conf[0])

            if cls_name == "phone":
                frame_has_phone = True
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(display_frame, f"PHONE {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            elif cls_name in ("drinking", "eating", "cigarette"):
                is_near_mouth = False
                if mouth_x > 0 and mouth_y > 0:
                    padding = 60
                    if (x1 - padding) <= mouth_x <= (x2 + padding) and (y1 - padding) <= mouth_y <= (y2 + padding):
                        is_near_mouth = True
                else:
                    is_near_mouth = True

                if is_near_mouth:
                    frame_has_ingestion = True
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                    cv2.putText(display_frame, f"INGESTION {conf_val:.2f}", (x1, y1 - 10), _HUD_FONT, 0.5, (0, 165, 255), 2, cv2.LINE_AA)

        phone_buffer.append(frame_has_phone)
        ingestion_buffer.append(frame_has_ingestion)
        detected_phone = sum(phone_buffer) >= int(0.3 * BUFFER_WINDOW)
        detected_ingestion = sum(ingestion_buffer) >= int(0.3 * BUFFER_WINDOW)

        # --- 4. HUD RENDER PHASE ---
        active_alerts = []
        if is_drowsy:
            active_alerts.append(("CRITICAL: DROWSINESS DETECTED", (0, 0, 255), 2))
        if detected_phone:
            if current_pitch < -5:
                active_alerts.append(("CRITICAL: TEXTING & DRIVING", (0, 0, 255), 2))
            else:
                active_alerts.append(("WARNING: CELLPHONE DISTRACTION", (0, 165, 255), 1))
        if is_distracted_gaze:
            active_alerts.append(("WARNING: DRIVER GAZE DISTRACTED", (0, 255, 255), 1))
        if detected_ingestion:
            active_alerts.append(("WARNING: INGESTION DISTRACTION", (0, 165, 255), 1))
        if is_yawning:
            active_alerts.append(("ADVISORY: DRIVER FATIGUE YAWN", (255, 255, 0), 0))

        render_hud(display_frame, active_alerts, current_pitch, current_yaw, current_mar, fps)
        out.write(display_frame)

        if frame_count % 50 == 0:
            print(f"Processed {frame_count}/{total_frames} frames...")

    cap.release()
    out.release()
    print(f"\n✅ Processing complete! Download your processed video from the Colab file browser: {output_path}")

process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Loading video: /content/headturn.mp4
Processing 551 frames...
Loading /content/yolo_dms.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CUDAExecutionProvider
Processed 50/551 frames...
Processed 100/551 frames...
Processed 150/551 frames...
Processed 200/551 frames...
Processed 250/551 frames...
Processed 300/551 frames...
Processed 350/551 frames...
Processed 400/551 frames...
Processed 450/551 frames...
Processed 500/551 frames...
Processed 550/551 frames...

✅ Processing complete! Download your processed video from the Colab file browser: /content/output.mp4
